# Sen1Floods11 final clean run: multi-seed LOEO, larger ensembles, train+valid fit (2 x T4)

Cleaned version of the main pipeline: only the four variants that matter (`S1`, `S1+ref`, `S1+dw`, `S1+ref+dw`), Dice loss, no discarded experiments.

**What it adds on top of the earlier runs (they are reused automatically if attached as a dataset)**
1. **Leave-one-event-out with 3 seeds** per variant (seed 0 is reused): per-seed macro IoU (mean +- sd), a joint paired bootstrap averaged over seeds, and the seed-ensembled LOEO result. This turns the main claim from one seed into three.
2. **Official split with 5 seeds x 3 encoders = 15 models per variant** (seeds 0-2 are reused): larger ensembles, same protocol as before (train split only, epochs chosen on valid, threshold from valid).
3. **Train+valid fit (non-standard protocol, reported separately):** `S1` and `S1+ref+dw`, 3 encoders x 2 seeds, trained on train+valid (341 chips) with fixed 3000 steps and final EMA weights; the threshold is taken from the train-only ensemble of the same variant, nothing is tuned on test. Use it as an "extra data" result, not as the benchmark number.

**Time (2 x T4):** about 9 h from scratch if attached results are reused (LOEO about 3.9 h for 88 new jobs, official split about 3.2 h for 24 new jobs, train+valid about 1.7 h). Phases are ordered and time-guarded: LOEO may use the first 55% of `DEADLINE_H`, the official split until 85%, train+valid until 100%. Every job writes its own result file, so a stopped run resumes when started again. **On a 4.5 h session set `DEADLINE_H=4.2`:** it finishes LOEO seed 1 (about 2 h), starts the extra official-split seeds, and mostly skips train+valid; run the rest later on the other account with the output attached.

**Honest expectations.** An ensemble of more seeds usually moves the test IoU by about +-0.01; the 90-chip test set cannot resolve that. The multi-seed LOEO is the number that matters for the paper. Reference/land-cover variants are multi-source/bi-temporal: do not tabulate them next to single-date published scores without saying so.

**Setup:** Kaggle GPU T4 x2, Internet on, attach your previous output as a dataset (cached `*_ref.npy`, `*_dw.npy`, terrain and `jobs_results/*.npz`), secrets `EE_PROJECT` and `EE_CREDENTIALS_JSON` only if something must be fetched.

In [1]:
import os, sys, glob, re, json, time, random, shutil, subprocess, warnings, io, copy, math
warnings.filterwarnings("ignore")
T0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "rasterio", "earthengine-api"], check=False)

import numpy as np, pandas as pd, torch, torch.nn.functional as F
import rasterio
import segmentation_models_pytorch as smp

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", DEV, "| torch", torch.__version__, "| smp", smp.__version__)

device: cuda | torch 2.11.0+cu130 | smp 0.6.0


In [2]:

CFG = dict(
    SMOKE=False, EE_PROJECT=None,
    VARIANTS={"S1": [], "S1+ref": ["ref"], "S1+dw": ["dw"], "S1+ref+dw": ["ref", "dw"]},
    LOSS={},                                              # everything: BCE + Dice
    LOEO_ENC="unet:resnet34", LOEO_STEPS=1500, LOEO_SEEDS=[0, 1, 2],
    FINAL_ENCODERS=["unet:resnet34", "unet:resnext50_32x4d", "unet:mit_b2"], FINAL_SEEDS=[0, 1, 2, 3, 4], FINAL_STEPS=3000,
    FULL_VARIANTS=["S1", "S1+ref+dw"], FULL_SEEDS=[0, 1],
    CROP=256, BS=24, LR=3e-4, WD=1e-4, EMA=0.995, EVAL_EVERY=250, ENC_WEIGHTS="imagenet", SIZE=512,
    EE_WORKERS=8, DEADLINE_H=12.0,
    OUT="/kaggle/working/out", ROOT="/kaggle/working/s1f11", REF="/kaggle/working/ref", TERR="/kaggle/working/terrain",
    DWX="/kaggle/working/dwx", CACHE="/kaggle/working/cache",
)
if CFG["SMOKE"]:
    CFG.update(LOEO_STEPS=30, FINAL_STEPS=30, FINAL_ENCODERS=CFG["FINAL_ENCODERS"][:1], FINAL_SEEDS=[0], LOEO_SEEDS=[0, 1], FULL_SEEDS=[0], EVAL_EVERY=15)
for k in ("OUT", "REF", "TERR", "DWX", "CACHE"): os.makedirs(CFG[k], exist_ok=True)
TERRAIN_ALL = ["jrc", "dem", "slope", "hand"]
def hours(): return (time.time() - T0) / 3600


In [3]:
# ---------- hand-labelled chips + official splits ----------
ROOT, B = CFG["ROOT"], "gs://sen1floods11/v1.1"
def sh(c):
    print("$", c); return subprocess.run(c, shell=True, capture_output=True, text=True)

if len(glob.glob(f"{ROOT}/S1Hand/*.tif")) < 446:
    if shutil.which("gsutil") is None: sh("pip install -q gsutil")
    os.makedirs(ROOT, exist_ok=True)
    sh(f"gsutil -m -q cp -r {B}/splits/flood_handlabeled {ROOT}/")
    for d in ["S1Hand", "LabelHand"]:
        print(sh(f"gsutil -m -q cp -r {B}/data/flood_events/HandLabeled/{d} {ROOT}/").stderr[-300:])

def find_csv(name):
    p = glob.glob(f"{ROOT}/**/{name}", recursive=True); assert p, f"missing {name}"; return p[0]
def read_split(name):
    df = pd.read_csv(find_csv(name), header=None)
    return [(f"{ROOT}/S1Hand/{a}", f"{ROOT}/LabelHand/{b}") for a, b in zip(df[0], df[1])]

SPLIT = {k: read_split(f"flood_{k}_data.csv") for k in ["train", "valid", "test", "bolivia"]}
SPLIT["bol"] = SPLIT.pop("bolivia")
ITEMS, IDX, pos = [], {}, 0
for k in ["train", "valid", "test", "bol"]:
    IDX[k] = np.arange(pos, pos + len(SPLIT[k])); ITEMS += SPLIT[k]; pos += len(SPLIT[k])
cid = lambda p: re.sub(r"_(S1Hand|LabelHand)\.tif$", "", os.path.basename(p))
IDS = [cid(a) for a, _ in ITEMS]
EVENT = np.array([i.split("_")[0] for i in IDS])
EVENTS = sorted(set(EVENT))
print({k: len(v) for k, v in IDX.items()}, "| events:", EVENTS)
assert len(ITEMS) == len(set(IDS)) and all(os.path.exists(a) and os.path.exists(b) for a, b in ITEMS)
if CFG["SMOKE"]: print("SMOKE: all chips are still used for terrain fetch / LOEO folds are limited below")

$ gsutil -m -q cp -r gs://sen1floods11/v1.1/splits/flood_handlabeled /kaggle/working/s1f11/
$ gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand /kaggle/working/s1f11/

$ gsutil -m -q cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand /kaggle/working/s1f11/

{'train': 252, 'valid': 89, 'test': 90, 'bol': 15} | events: [np.str_('Bolivia'), np.str_('Ghana'), np.str_('India'), np.str_('Mekong'), np.str_('Nigeria'), np.str_('Pakistan'), np.str_('Paraguay'), np.str_('Somalia'), np.str_('Spain'), np.str_('Sri-Lanka'), np.str_('USA')]


In [4]:

# ---------- pre-event Sentinel-1 reference per chip (Earth Engine) ----------
# event -> (S1 acquisition date, orbit pass, relative orbit), from Sen1Floods11_Metadata.geojson (Mekong = Cambodia entry)
META = {
    "Bolivia": ("2018-02-15", "DESCENDING", 156), "Ghana": ("2018-09-18", "ASCENDING", 147), "India": ("2016-08-12", "DESCENDING", 77),
    "Mekong": ("2018-08-05", "ASCENDING", 26), "Nigeria": ("2018-09-21", "ASCENDING", 103), "Pakistan": ("2017-06-28", "DESCENDING", 5),
    "Paraguay": ("2018-10-31", "DESCENDING", 68), "Somalia": ("2018-05-07", "ASCENDING", 116), "Spain": ("2019-09-17", "DESCENDING", 110),
    "Sri-Lanka": ("2017-05-30", "DESCENDING", 19), "USA": ("2019-05-22", "ASCENDING", 136),
}
assert set(EVENTS) <= set(META), f"events without metadata: {set(EVENTS) - set(META)}"
REFB = ["vv_rec", "vh_rec", "n_rec", "vv_yr", "vh_yr", "n_yr", "sdv_yr", "sdh_yr"]
WIN = {"rec": (132, 12), "yr": (377, 12)}       # (start, end) in days before the event image
def ref_path(i): return f"{CFG['REF']}/{i}_ref.npy"

missing = [i for i in IDS if not os.path.exists(ref_path(i))]
if missing:
    hit = glob.glob(f"/kaggle/input/**/{IDS[0]}_ref.npy", recursive=True)
    if hit:
        for f in glob.glob(os.path.dirname(hit[0]) + "/*_ref.npy"): shutil.copy(f, CFG["REF"])
        missing = [i for i in IDS if not os.path.exists(ref_path(i))]
        print("copied reference cache from", os.path.dirname(hit[0]), "| still missing", len(missing))

if missing:
    import ee
    from concurrent.futures import ThreadPoolExecutor
    proj, key, cred_json = CFG["EE_PROJECT"], None, None
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        try: key = us.get_secret("EE_SERVICE_ACCOUNT_JSON")
        except Exception: key = None
        try: cred_json = us.get_secret("EE_CREDENTIALS_JSON")
        except Exception: cred_json = None
        try: proj = proj or us.get_secret("EE_PROJECT")
        except Exception: pass
    except Exception: pass
    hv = "https://earthengine-highvolume.googleapis.com"
    def ee_init(creds=None):
        last = None
        for url in (hv, None):
            try:
                if url: ee.Initialize(creds, project=proj, opt_url=url)
                else: ee.Initialize(creds, project=proj)
                return
            except Exception as e: last = e
        raise last
    try:
        if key:                                        # service-account key (blocked by some org policies)
            info = json.loads(key); proj = proj or info.get("project_id")
            ee_init(ee.ServiceAccountCredentials(info["client_email"], key_data=key))
        elif cred_json:                                # refresh token from your own account (no key creation needed)
            from google.oauth2.credentials import Credentials
            d = json.loads(cred_json)
            ee_init(Credentials(None, refresh_token=d["refresh_token"], token_uri="https://oauth2.googleapis.com/token",
                                client_id=d.get("client_id", ee.oauth.CLIENT_ID), client_secret=d.get("client_secret", ee.oauth.CLIENT_SECRET),
                                scopes=ee.oauth.SCOPES))
        else:                                          # interactive: works in an interactive session, not in Save & Run All
            try: ee_init()
            except Exception:
                ee.Authenticate(auth_mode="notebook"); ee_init()
    except Exception as e:
        raise RuntimeError("Earth Engine init failed (%s). Set EE_PROJECT and use EE_CREDENTIALS_JSON, or run this cell interactively once, or attach a terrain cache dataset." % e)


    S1C = (ee.ImageCollection("COPERNICUS/S1_GRD").filter(ee.Filter.eq("instrumentMode", "IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")).select(["VV", "VH"]))
    BLANK = ee.Image.constant([0, 0]).rename(["VV", "VH"]).float().updateMask(ee.Image.constant(0))

    def ref_image(event, bbox, use_rel):
        d, orbit, rel = META[event]
        c = S1C.filterBounds(ee.Geometry.Rectangle(bbox, "EPSG:4326", False)).filter(ee.Filter.eq("orbitProperties_pass", orbit))
        if use_rel: c = c.filter(ee.Filter.eq("relativeOrbitNumber_start", int(rel)))
        t = ee.Date(d); out = []
        for tag, (a, b) in WIN.items():
            cc = c.filterDate(t.advance(-a, "day"), t.advance(-b, "day")).merge(ee.ImageCollection([BLANK]))   # never empty
            out += [cc.median().rename([f"vv_{tag}", f"vh_{tag}"]), cc.select("VV").count().rename(f"n_{tag}")]
            if tag == "yr": out += [cc.reduce(ee.Reducer.stdDev()).rename(["sdv_yr", "sdh_yr"])]
        return ee.Image.cat(out).unmask(-999).toFloat()

    def chip_geo(i):
        with rasterio.open(ITEMS[IDS.index(i)][0]) as s: t, h, w = s.transform, s.height, s.width
        x0, y0, x1, y1 = t.c, t.f, t.c + w * t.a, t.f + h * t.e
        grid = dict(dimensions=dict(width=w, height=h), crsCode="EPSG:4326",
                    affineTransform=dict(scaleX=t.a, shearX=t.b, translateX=t.c, shearY=t.d, scaleY=t.e, translateY=t.f))
        return grid, [min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)]

    def get_ref(i, use_rel):
        grid, bbox = chip_geo(i)
        raw = ee.data.computePixels(dict(expression=ref_image(EVENT[IDS.index(i)], bbox, use_rel), fileFormat="NPY", grid=grid))
        a = np.load(io.BytesIO(raw), allow_pickle=False)
        return np.stack([a[n] for n in REFB], 0) if a.dtype.names else np.moveaxis(a, -1, 0)

    def fetch_one(i):
        if os.path.exists(ref_path(i)): return i, None
        err = None
        for k in range(5):
            try:
                a = get_ref(i, True)
                if a[5].max() <= 0:                      # no same-relative-orbit images: fall back to same pass only
                    a = get_ref(i, False)
                np.save(ref_path(i), a.astype(np.float16)); return i, None
            except Exception as e:
                err = str(e)[:200]; time.sleep(2 ** k)
        return i, err

    i0, e0 = fetch_one(missing[0]); assert e0 is None, f"first EE fetch failed: {e0}"
    a0 = np.load(ref_path(i0)).astype(np.float32)
    print("first chip", i0, "| ref shape", a0.shape, "| median n_recent / n_year:", float(np.median(a0[2])), float(np.median(a0[5])),
          "| ref VV median (dB):", float(np.median(a0[0][a0[0] > -900])) if (a0[0] > -900).any() else "none")
    t1 = time.time(); fails = []
    with ThreadPoolExecutor(CFG["EE_WORKERS"]) as ex:
        for n, (i, e) in enumerate(ex.map(fetch_one, missing[1:]), 1):
            if e: fails.append((i, e))
            if n % 50 == 0: print(f"  fetched {n}/{len(missing) - 1} | {time.time() - t1:.0f}s | failures {len(fails)}")
    print("EE fetch done in %.0fs, failures: %d" % (time.time() - t1, len(fails)), fails[:3])
    assert len(fails) < 0.02 * len(IDS), "too many EE failures"

REF = {i: (np.load(ref_path(i)).astype(np.float32) if os.path.exists(ref_path(i)) else None) for i in IDS}
rows = []
for e in EVENTS:
    ch = [REF[i] for i, ev in zip(IDS, EVENT) if ev == e and REF[i] is not None]
    rows.append(dict(event=e, chips=int((EVENT == e).sum()), with_ref=len(ch),
                     median_n_recent=float(np.median([np.median(r[2]) for r in ch])) if ch else 0.0,
                     median_n_year=float(np.median([np.median(r[5]) for r in ch])) if ch else 0.0,
                     frac_year_ref_pixels=float(np.mean([(r[3] > -900).mean() for r in ch])) if ch else 0.0))
REF_Q = pd.DataFrame(rows); print(REF_Q.round(3).to_string(index=False))
assert (REF_Q["frac_year_ref_pixels"] > 0.5).all(), "an event has no usable one-year reference; check the orbit metadata/printed table"


copied reference cache from /kaggle/input/datasets/<kaggle-user>/<dataset>/ref | still missing 0
    event  chips  with_ref  median_n_recent  median_n_year  frac_year_ref_pixels
  Bolivia     15        15             10.0           29.0                 0.958
    Ghana     53        53             14.0           34.0                 0.994
    India     68        68              4.0           12.0                 0.987
   Mekong     30        30             13.0           33.0                 1.000
  Nigeria     18        18              8.0           28.0                 0.972
 Pakistan     28        28             10.0           10.0                 1.000
 Paraguay     67        67             10.0           30.0                 1.000
  Somalia     26        26             10.0           27.0                 1.000
    Spain     30        30             19.0           78.0                 1.000
Sri-Lanka     42        42              8.0           13.0                 1.000
      USA   

In [5]:
# ---------- terrain from Earth Engine, pixel-aligned to each chip ----------
def terrain_path(i): return f"{CFG['TERR']}/{i}.npy"

def cached_terrain_dirs():
    hit = glob.glob(f"/kaggle/input/**/{IDS[0]}.npy", recursive=True)
    return os.path.dirname(hit[0]) + "/" if hit else None

missing = [i for i in IDS if not os.path.exists(terrain_path(i))]
if missing:
    alt = cached_terrain_dirs()
    if alt:
        for f in glob.glob(alt + "*.npy"): shutil.copy(f, CFG["TERR"])
        missing = [i for i in IDS if not os.path.exists(terrain_path(i))]
        print("copied terrain cache from", alt, "| still missing", len(missing))

if missing:
    import ee
    from concurrent.futures import ThreadPoolExecutor
    proj, key, cred_json = CFG["EE_PROJECT"], None, None
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        try: key = us.get_secret("EE_SERVICE_ACCOUNT_JSON")
        except Exception: key = None
        try: cred_json = us.get_secret("EE_CREDENTIALS_JSON")
        except Exception: cred_json = None
        try: proj = proj or us.get_secret("EE_PROJECT")
        except Exception: pass
    except Exception: pass
    hv = "https://earthengine-highvolume.googleapis.com"
    def ee_init(creds=None):
        last = None
        for url in (hv, None):
            try:
                if url: ee.Initialize(creds, project=proj, opt_url=url)
                else: ee.Initialize(creds, project=proj)
                return
            except Exception as e: last = e
        raise last
    try:
        if key:                                        # service-account key (blocked by some org policies)
            info = json.loads(key); proj = proj or info.get("project_id")
            ee_init(ee.ServiceAccountCredentials(info["client_email"], key_data=key))
        elif cred_json:                                # refresh token from your own account (no key creation needed)
            from google.oauth2.credentials import Credentials
            d = json.loads(cred_json)
            ee_init(Credentials(None, refresh_token=d["refresh_token"], token_uri="https://oauth2.googleapis.com/token",
                                client_id=d.get("client_id", ee.oauth.CLIENT_ID), client_secret=d.get("client_secret", ee.oauth.CLIENT_SECRET),
                                scopes=ee.oauth.SCOPES))
        else:                                          # interactive: works in an interactive session, not in Save & Run All
            try: ee_init()
            except Exception:
                ee.Authenticate(auth_mode="notebook"); ee_init()
    except Exception as e:
        raise RuntimeError("Earth Engine init failed (%s). Set EE_PROJECT and use EE_CREDENTIALS_JSON, or run this cell interactively once, or attach a terrain cache dataset." % e)

    dem = ee.Image("NASA/NASADEM_HGT/001").select("elevation")
    img = ee.Image.cat([
        ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("seasonality").rename("jrc"),
        dem.rename("dem"),
        ee.Terrain.slope(dem).rename("slope"),
        ee.ImageCollection("users/gena/global-hand/hand-100").mosaic().rename("hand"),
    ]).unmask(0).toFloat()

    def chip_grid(i):
        with rasterio.open(ITEMS[IDS.index(i)][0]) as s:
            t, h, w = s.transform, s.height, s.width
        return dict(dimensions=dict(width=w, height=h), crsCode="EPSG:4326",
                    affineTransform=dict(scaleX=t.a, shearX=t.b, translateX=t.c, shearY=t.d, scaleY=t.e, translateY=t.f))

    def fetch_one(i):
        if os.path.exists(terrain_path(i)): return i, None
        err = None
        for k in range(5):
            try:
                raw = ee.data.computePixels(dict(expression=img, fileFormat="NPY", grid=chip_grid(i)))
                a = np.load(io.BytesIO(raw), allow_pickle=False)
                a = np.stack([a[n] for n in TERRAIN_ALL], 0) if a.dtype.names else np.moveaxis(a, -1, 0)
                np.save(terrain_path(i), a.astype(np.float16)); return i, None
            except Exception as e:
                err = str(e)[:200]; time.sleep(2 ** k)
        return i, err

    # test one chip first (fail fast), then the rest
    i0, e0 = fetch_one(missing[0]); assert e0 is None, f"first EE fetch failed: {e0}"
    a0 = np.load(terrain_path(i0)).astype(np.float32)
    print("first chip", i0, "shape", a0.shape, "| per-layer min/max:", [(TERRAIN_ALL[j], float(a0[j].min()), float(a0[j].max())) for j in range(4)])
    t1 = time.time(); fails = []
    with ThreadPoolExecutor(CFG["EE_WORKERS"]) as ex:
        for n, (i, e) in enumerate(ex.map(fetch_one, missing[1:]), 1):
            if e: fails.append((i, e))
            if n % 50 == 0: print(f"  fetched {n}/{len(missing) - 1} | {time.time() - t1:.0f}s | failures {len(fails)}")
    print("EE fetch done in %.0fs, failures: %d" % (time.time() - t1, len(fails)), fails[:3])
    assert len(fails) < 0.02 * len(IDS), "too many EE failures"

TERR = {}
for n, i in enumerate(IDS):
    p = terrain_path(i)
    TERR[i] = np.load(p).astype(np.float32) if os.path.exists(p) else None
nm = sum(v is None for v in TERR.values())
print("terrain chips loaded:", len(TERR) - nm, "| missing:", nm)
assert nm < 0.02 * len(IDS)

copied terrain cache from /kaggle/input/datasets/<kaggle-user>/<dataset>/terrain/ | still missing 0
terrain chips loaded: 446 | missing: 0


In [6]:

# ---------- Dynamic World land-cover probabilities (pre-event year mean) + ERA5-Land weather scalars ----------
DW_BANDS = ["water", "trees", "grass", "flooded_vegetation", "crops", "shrub_and_scrub", "built", "bare", "snow_and_ice"]
WX_NAMES = ["p1", "p7", "p30", "p90", "sm0", "sm30"]          # precipitation mm before event (1/7/30/90 d), soil moisture (event day, 30 d mean)
def dw_path(i): return f"{CFG['DWX']}/{i}_dw.npy"
def wx_path(i): return f"{CFG['DWX']}/{i}_wx.npy"
missing_d = [i for i in IDS if not (os.path.exists(dw_path(i)) and os.path.exists(wx_path(i)))]
if missing_d:
    hit = glob.glob(f"/kaggle/input/**/{IDS[0]}_dw.npy", recursive=True)
    if hit:
        for f in glob.glob(os.path.dirname(hit[0]) + "/*_dw.npy") + glob.glob(os.path.dirname(hit[0]) + "/*_wx.npy"): shutil.copy(f, CFG["DWX"])
        missing_d = [i for i in IDS if not (os.path.exists(dw_path(i)) and os.path.exists(wx_path(i)))]
        print("copied dw/wx cache from", os.path.dirname(hit[0]), "| still missing", len(missing_d))
if missing_d:
    import ee
    from concurrent.futures import ThreadPoolExecutor
    proj, key, cred_json = CFG["EE_PROJECT"], None, None
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        try: key = us.get_secret("EE_SERVICE_ACCOUNT_JSON")
        except Exception: key = None
        try: cred_json = us.get_secret("EE_CREDENTIALS_JSON")
        except Exception: cred_json = None
        try: proj = proj or us.get_secret("EE_PROJECT")
        except Exception: pass
    except Exception: pass
    hv = "https://earthengine-highvolume.googleapis.com"
    def ee_init(creds=None):
        last = None
        for url in (hv, None):
            try:
                if url: ee.Initialize(creds, project=proj, opt_url=url)
                else: ee.Initialize(creds, project=proj)
                return
            except Exception as e: last = e
        raise last
    try:
        if key:                                        # service-account key (blocked by some org policies)
            info = json.loads(key); proj = proj or info.get("project_id")
            ee_init(ee.ServiceAccountCredentials(info["client_email"], key_data=key))
        elif cred_json:                                # refresh token from your own account (no key creation needed)
            from google.oauth2.credentials import Credentials
            d = json.loads(cred_json)
            ee_init(Credentials(None, refresh_token=d["refresh_token"], token_uri="https://oauth2.googleapis.com/token",
                                client_id=d.get("client_id", ee.oauth.CLIENT_ID), client_secret=d.get("client_secret", ee.oauth.CLIENT_SECRET),
                                scopes=ee.oauth.SCOPES))
        else:                                          # interactive: works in an interactive session, not in Save & Run All
            try: ee_init()
            except Exception:
                ee.Authenticate(auth_mode="notebook"); ee_init()
    except Exception as e:
        raise RuntimeError("Earth Engine init failed (%s). Set EE_PROJECT and use EE_CREDENTIALS_JSON, or run this cell interactively once, or attach a terrain cache dataset." % e)


    BLANK9 = ee.Image.constant([0] * 9).rename(DW_BANDS).float().updateMask(ee.Image.constant(0))
    def dwx_image(event, bbox):
        d = META[event][0]; t = ee.Date(d); geom = ee.Geometry.Rectangle(bbox, "EPSG:4326", False)
        dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(geom).filterDate(t.advance(-377, "day"), t.advance(-12, "day"))
              .select(DW_BANDS).map(lambda im: im.float()).merge(ee.ImageCollection([BLANK9])).mean())
        era = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        def psum(n): return era.filterDate(t.advance(-n, "day"), t).select("total_precipitation_sum").sum().multiply(1000).rename(f"p{n}")
        sm0 = era.filterDate(t.advance(-1, "day"), t.advance(1, "day")).select("volumetric_soil_water_layer_1").mean().rename("sm0")
        sm30 = era.filterDate(t.advance(-30, "day"), t).select("volumetric_soil_water_layer_1").mean().rename("sm30")
        return ee.Image.cat([dw, psum(1), psum(7), psum(30), psum(90), sm0, sm30]).unmask(0).toFloat()
    def geo_of(i):
        with rasterio.open(ITEMS[IDS.index(i)][0]) as s: t, h, w = s.transform, s.height, s.width
        x0, y0, x1, y1 = t.c, t.f, t.c + w * t.a, t.f + h * t.e
        grid = dict(dimensions=dict(width=w, height=h), crsCode="EPSG:4326",
                    affineTransform=dict(scaleX=t.a, shearX=t.b, translateX=t.c, shearY=t.d, scaleY=t.e, translateY=t.f))
        return grid, [min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)]
    def fetch_dwx(i):
        if os.path.exists(dw_path(i)) and os.path.exists(wx_path(i)): return i, None
        err = None
        for k in range(5):
            try:
                grid, bbox = geo_of(i)
                raw = ee.data.computePixels(dict(expression=dwx_image(EVENT[IDS.index(i)], bbox), fileFormat="NPY", grid=grid))
                a = np.load(io.BytesIO(raw), allow_pickle=False)
                get = (lambda n: a[n]) if a.dtype.names else (lambda n, a=np.moveaxis(a, -1, 0): a[(DW_BANDS + WX_NAMES).index(n)])
                np.save(dw_path(i), np.stack([get(n) for n in DW_BANDS], 0).astype(np.float16))
                np.save(wx_path(i), np.array([np.median(get(n)) for n in WX_NAMES], np.float32)); return i, None
            except Exception as e:
                err = str(e)[:200]; time.sleep(2 ** k)
        return i, err
    i0, e0 = fetch_dwx(missing_d[0]); assert e0 is None, f"first DW/ERA5 fetch failed: {e0}"
    print("first chip", i0, "| dw mean per class:", np.load(dw_path(i0)).astype(np.float32).mean((1, 2)).round(3), "| weather:", dict(zip(WX_NAMES, np.load(wx_path(i0)).round(3))))
    t1 = time.time(); fails = []
    with ThreadPoolExecutor(CFG["EE_WORKERS"]) as ex:
        for n, (i, e) in enumerate(ex.map(fetch_dwx, missing_d[1:]), 1):
            if e: fails.append((i, e))
            if n % 50 == 0: print(f"  fetched {n}/{len(missing_d) - 1} | {time.time() - t1:.0f}s | failures {len(fails)}")
    print("DW/ERA5 fetch done in %.0fs, failures: %d" % (time.time() - t1, len(fails)), fails[:3])
    assert len(fails) < 0.02 * len(IDS), "too many EE failures"
print("dw files:", sum(os.path.exists(dw_path(i)) for i in IDS), "| wx files:", sum(os.path.exists(wx_path(i)) for i in IDS), "of", len(IDS))


copied dw/wx cache from /kaggle/input/datasets/<kaggle-user>/<dataset>/dwx | still missing 0
dw files: 446 | wx files: 446 of 446


In [7]:

# ---------- build the shared on-disk cache the GPU workers memory-map ----------
from scipy.ndimage import uniform_filter
S = CFG["SIZE"]; N = len(ITEMS); CACHE = CFG["CACHE"]; NCH = 20
GROUPS = {"base": [0, 1, 2], "ref": [3, 4, 5, 6], "rel": [7, 8, 9, 10], "dw": list(range(11, 20))}
def read_tif(p):
    with rasterio.open(p) as s: return s.read().astype(np.float32)

def build_chip(n):
    a, b = ITEMS[n]; s1 = read_tif(a)[:2]
    s1[0] = np.where(np.isfinite(s1[0]), s1[0], -10.0); s1[1] = np.where(np.isfinite(s1[1]), s1[1], -17.0)   # nodata -> typical land
    s1 = np.clip(s1, -35.0, 5.0)
    x = np.full((NCH, S, S), np.nan, np.float32)
    x[0], x[1], x[2] = s1[0], s1[1], s1[0] - s1[1]
    for pol in (0, 1):                       # relative backscatter
        x[7 + pol] = s1[pol] - np.median(s1[pol]); x[9 + pol] = s1[pol] - uniform_filter(s1[pol], size=min(65, S - 1), mode="reflect")
    r = REF[IDS[n]]                          # reference bands: 3 vv_yr, 4 vh_yr, 6 sd_vv, 7 sd_vh
    if r is not None:
        for ch, k, pol in ((3, 3, 0), (4, 4, 1)):
            x[ch] = np.where(r[k] > -900, s1[pol] - np.clip(r[k], -35.0, 5.0), np.nan)
        for ch, k, sk, pol in ((5, 3, 6, 0), (6, 4, 7, 1)):
            ok = (r[k] > -900) & (r[sk] > -900)
            x[ch] = np.where(ok, np.clip((s1[pol] - np.clip(r[k], -35.0, 5.0)) / np.maximum(r[sk], 1.0), -6.0, 6.0), np.nan)
    if os.path.exists(dw_path(IDS[n])): x[11:20] = np.load(dw_path(IDS[n])).astype(np.float32)
    return x, read_tif(b)[0].astype(np.int8)

tr = IDX["train"]; sm, sq, ct = np.zeros(NCH), np.zeros(NCH), np.zeros(NCH)
for n in tr:                                 # pass 1: per-channel statistics on official-train chips only
    x, _ = build_chip(n); ok = np.isfinite(x); xv = np.where(ok, x, 0.0).astype(np.float64)
    sm += xv.sum((1, 2)); sq += (xv ** 2).sum((1, 2)); ct += ok.sum((1, 2))
MEAN = sm / np.maximum(ct, 1); STD = np.sqrt(np.maximum(sq / np.maximum(ct, 1) - MEAN ** 2, 0)) + 1e-6
XM = np.lib.format.open_memmap(f"{CACHE}/X.npy", mode="w+", dtype=np.float16, shape=(N, NCH, S, S))
Y = np.zeros((N, S, S), np.int8)
for n in range(N):                           # pass 2: normalise, NaN -> 0, write fp16
    x, y = build_chip(n)
    XM[n] = np.clip(np.nan_to_num((x - MEAN[:, None, None]) / STD[:, None, None], nan=0.0), -30, 30).astype(np.float16); Y[n] = y
XM.flush(); del XM
np.save(f"{CACHE}/Y.npy", Y)
WXR = np.stack([np.load(wx_path(i)) if os.path.exists(wx_path(i)) else np.full(len(WX_NAMES), np.nan, np.float32) for i in IDS]).astype(np.float64)
wm, ws = np.nanmean(WXR[IDX["train"]], 0), np.nanstd(WXR[IDX["train"]], 0) + 1e-6
np.save(f"{CACHE}/wx.npy", np.nan_to_num((WXR - wm) / ws, nan=0.0).astype(np.float32))
json.dump(dict(ids=IDS, event=[str(e) for e in EVENT], idx={k: v.tolist() for k, v in IDX.items()}, S=S, nch=NCH,
               std_vv=float(STD[0]), std_vh=float(STD[1]), groups=GROUPS), open(f"{CACHE}/meta.json", "w"))
del REF
print("cache:", N, "chips x", NCH, "channels | X.npy %.1f GB" % (os.path.getsize(f"{CACHE}/X.npy") / 1e9), "| water share", float((Y == 1).sum() / (Y >= 0).sum()))
print("weather scalars per event (median over chips):\n", pd.DataFrame(WXR, columns=WX_NAMES).assign(event=EVENT).groupby("event").median().round(3).to_string())


cache: 446 chips x 20 channels | X.npy 4.7 GB | water share 0.1060136033959036
weather scalars per event (median over chips):
                p1       p7      p30       p90    sm0   sm30
event                                                      
Bolivia    13.879   77.458  289.704   926.326  0.475  0.468
Ghana       0.574   43.303  336.366   747.327  0.408  0.424
India       4.185   78.767  430.510  1351.091  0.474  0.478
Mekong      7.246   50.941  291.053   705.439  0.416  0.413
Nigeria     2.913   20.829  228.856   628.820  0.405  0.414
Pakistan    8.950   23.346   40.079    66.297  0.276  0.142
Paraguay    0.232   59.814  234.403   472.583  0.365  0.396
Somalia     3.251   47.235  198.671   218.394  0.360  0.319
Spain       1.240  155.493  174.330   178.397  0.364  0.222
Sri-Lanka   0.448    6.125   52.434   214.457  0.177  0.211
USA        38.347   83.445  262.692   423.019  0.414  0.363


In [8]:

WORKER_SRC = r"""
import os, sys, json, time, copy, random, traceback, threading, queue
import numpy as np
import torch, torch.nn.functional as F
import segmentation_models_pytorch as smp

X = Y = WX = META = None
EVENT = None; S = 512

def init(cache):
    global X, Y, WX, META, EVENT, S
    X = np.load(cache + "/X.npy", mmap_mode="r"); Y = np.load(cache + "/Y.npy"); WX = np.load(cache + "/wx.npy")
    META = json.load(open(cache + "/meta.json")); EVENT = np.array(META["event"]); S = META["S"]

def make_batch(idx, chans, use_wx, bs, crop, rng):
    xs, ys = [], []
    for i in rng.choice(idx, bs):
        r = int(rng.integers(0, S - crop + 1)); c = int(rng.integers(0, S - crop + 1))
        x = np.asarray(X[i][chans, r:r + crop, c:c + crop], dtype=np.float32); y = Y[i, r:r + crop, c:c + crop]
        if rng.random() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
        if rng.random() < .5: x, y = x[:, ::-1], y[::-1]
        k = int(rng.integers(4))
        if k: x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
        x = np.ascontiguousarray(x); sh = rng.uniform(-1, 1)
        x[0] += sh / META["std_vv"]; x[1] += sh / META["std_vh"]          # +-1 dB backscatter shift (VV-VH and references unchanged)
        if use_wx: x = np.concatenate([x, np.broadcast_to(WX[i][:, None, None], (WX.shape[1], crop, crop))], 0)
        xs.append(x); ys.append(np.ascontiguousarray(y))
    return torch.from_numpy(np.stack(xs)), torch.from_numpy(np.stack(ys).astype(np.float32))[:, None]

def lovasz_grad(gt):
    gts = gt.sum(); inter = gts - gt.cumsum(0); union = gts + (1 - gt).cumsum(0)
    j = 1.0 - inter / union
    if len(gt) > 1: j[1:] = j[1:] - j[:-1]
    return j

def lovasz_hinge_flat(logits, labels):
    signs = 2.0 * labels - 1.0; errors = 1.0 - logits * signs
    es, perm = torch.sort(errors, dim=0, descending=True)
    return torch.dot(F.relu(es), lovasz_grad(labels[perm]))

def masked_loss(logit, y, kind):
    m = (y >= 0).float(); t = (y == 1).float()
    bce = (F.binary_cross_entropy_with_logits(logit, t, reduction="none") * m).sum() / m.sum().clamp(min=1)
    p = torch.sigmoid(logit)
    if kind == "bce_tversky":
        tp = (p * t * m).sum(); fp = (p * (1 - t) * m).sum(); fn = ((1 - p) * t * m).sum()
        return bce + (1 - (tp + 1) / (tp + 0.7 * fp + 0.3 * fn + 1))
    if kind == "bce_lovasz":
        mb = m.bool()
        return bce + lovasz_hinge_flat(logit[mb], t[mb])
    pm = p * m; inter = (pm * t).sum((1, 2, 3)); den = pm.sum((1, 2, 3)) + (t * m).sum((1, 2, 3))
    return bce + (1 - ((2 * inter + 1) / (den + 1)).mean())

class EMA:
    def __init__(s, model, d):
        s.m = copy.deepcopy(model).eval(); s.d = d
        for p in s.m.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(s, model, step):
        d = min(s.d, (1 + step) / (10 + step))
        for e, p in zip(s.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(p.detach(), alpha=1 - d)
            else: e.copy_(p)

def make_model(spec, in_ch, weights):
    arch, enc = spec.split(":")
    cls = {"unet": smp.Unet, "upp": smp.UnetPlusPlus}[arch]
    return cls(enc, encoder_weights=(weights if weights != "none" else None), in_channels=in_ch, classes=1).cuda()

@torch.no_grad()
def predict(model, idx, chans, use_wx, tta, bs=8):
    model.eval(); out = []
    flips = [[], [3], [2], [2, 3]] if tta else [[]]
    for i in range(0, len(idx), bs):
        ii = list(idx[i:i + bs])
        x = np.stack([np.asarray(X[j][chans], dtype=np.float32) for j in ii])
        if use_wx: x = np.concatenate([x, np.stack([np.broadcast_to(WX[j][:, None, None], (WX.shape[1], S, S)) for j in ii]).astype(np.float32)], 1)
        x = torch.from_numpy(x).cuda(); ps = []
        for f in flips:
            xi = torch.flip(x, f) if f else x
            with torch.autocast("cuda", dtype=torch.float16):
                o = torch.sigmoid(model(xi).float())
            ps.append(torch.flip(o, f) if f else o)
        out.append(torch.stack(ps).mean(0)[:, 0].cpu().numpy().astype(np.float16))
    return np.concatenate(out)

def iou_pooled(prob, y, thr=0.5):
    m = y >= 0; pr = (prob >= thr) & m; t = (y == 1) & m
    return float((pr & t).sum() / max(((pr & t) | (pr & ~t) | (~pr & t)).sum(), 1))

def producer(q, stop, idx, chans, use_wx, bs, crop, seed, n):
    rng = np.random.default_rng(seed)
    try:
        for _ in range(n):
            if stop.is_set(): return
            q.put(make_batch(idx, chans, use_wx, bs, crop, rng))
    except Exception as e:
        q.put(e)

def train(job, cfg, train_idx, val_idx, tag):
    seed = job["seed"]; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    chans, use_wx, steps = job["chans"], job["wx"], job["steps"]
    in_ch = len(chans) + (WX.shape[1] if use_wx else 0)
    model = make_model(job["enc"], in_ch, cfg["ENC_WEIGHTS"]); ema = EMA(model, cfg["EMA"])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WD"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["LR"], total_steps=steps, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler()
    q = queue.Queue(maxsize=6); stop = threading.Event()
    th = threading.Thread(target=producer, args=(q, stop, train_idx, chans, use_wx, cfg["BS"], min(cfg["CROP"], S), seed, steps), daemon=True); th.start()
    best, best_sd, t0, run, every = -1.0, None, time.time(), 0.0, cfg["EVAL_EVERY"]
    for step in range(1, steps + 1):
        item = q.get()
        if isinstance(item, Exception): raise item
        x, y = item[0].cuda(non_blocking=True), item[1].cuda(non_blocking=True)
        model.train()
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x)
        loss = masked_loss(out.float(), y, job["loss"])
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(model, step)
        run += loss.item()
        if step % every == 0 or step == steps:
            msg = "[%s] step %d/%d loss %.4f (%.0fs)" % (tag, step, steps, run / min(every, step), time.time() - t0); run = 0.0
            if val_idx is not None:
                v = iou_pooled(predict(ema.m, val_idx, chans, use_wx, False).astype(np.float32), Y[val_idx])
                msg += " valid IoU(EMA) %.4f" % v
                if v > best: best, best_sd = v, {k: t.detach().cpu().clone() for k, t in ema.m.state_dict().items()}
            print(msg, flush=True)
    stop.set()
    if best_sd is not None: ema.m.load_state_dict(best_sd)
    del model, opt; torch.cuda.empty_cache()
    return ema.m, best

def run_job(job, cfg):
    tag = job["id"]
    if job["kind"] == "loeo":
        held = np.where(EVENT == job["held"])[0]; tr = np.where(EVENT != job["held"])[0]
        m, _ = train(job, cfg, tr, None, tag)
        p = predict(m, held, job["chans"], job["wx"], False)
        return dict(idx=held, p=p)
    if job["kind"] == "final_full":      # non-standard protocol: train on train+valid, final EMA weights, no epoch selection
        tr = np.array(META["idx"]["train"] + META["idx"]["valid"])
        m, _ = train(job, cfg, tr, None, tag)
        res = {k: predict(m, np.array(META["idx"][k]), job["chans"], job["wx"], True) for k in ("valid", "test", "bol")}
        res["best_valid"] = np.array(-1.0); return res
    tr = np.array(META["idx"]["train"]); va = np.array(META["idx"]["valid"])
    m, best = train(job, cfg, tr, va, tag)
    res = {k: predict(m, np.array(META["idx"][k]), job["chans"], job["wx"], True) for k in ("valid", "test", "bol")}
    res["best_valid"] = np.array(best); return res

def main():
    gpu, jobs_path, res_dir, cache, deadline = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], float(sys.argv[5])
    init(cache); spec = json.load(open(jobs_path)); cfg = spec["cfg"]
    print("worker on GPU", gpu, "| jobs", len(spec["jobs"]), flush=True)
    for job in spec["jobs"]:
        if time.time() > deadline: print("deadline reached, stopping", flush=True); break
        rp = res_dir + "/" + job["id"] + ".npz"; lk = res_dir + "/" + job["id"] + ".lock"
        if os.path.exists(rp): continue
        try: os.close(os.open(lk, os.O_CREAT | os.O_EXCL))
        except FileExistsError: continue
        try:
            t0 = time.time(); res = run_job(job, cfg)
            np.savez(rp + ".tmp.npz", **res); os.replace(rp + ".tmp.npz", rp)
            print("==> done %s in %.0fs" % (job["id"], time.time() - t0), flush=True)
        except Exception:
            traceback.print_exc(); sys.stdout.flush()
        finally:
            if os.path.exists(lk): os.remove(lk)

if __name__ == "__main__":
    main()
"""
WORKER = "/kaggle/working/worker.py"
open(WORKER, "w").write(WORKER_SRC)
import ast as _ast; _ast.parse(WORKER_SRC)

RES = f"{CFG['OUT']}/jobs_results"; os.makedirs(RES, exist_ok=True)
for f in glob.glob("/kaggle/input/**/jobs_results/*.npz", recursive=True):      # results of an earlier run attached as a dataset are reused
    if not f.endswith(".tmp.npz") and not os.path.exists(f"{RES}/{os.path.basename(f)}"): shutil.copy(f, RES)
print("job results available at start:", len(glob.glob(f"{RES}/*.npz")))
def ngpu():
    out = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout
    return max(1, sum(l.startswith("GPU") for l in out.splitlines()))
NG = ngpu(); print("GPUs visible:", NG, "(Kaggle: choose 'GPU T4 x2' to use both)")
WCFG = {k: CFG[k] for k in ("CROP", "BS", "LR", "WD", "EMA", "EVAL_EVERY", "ENC_WEIGHTS")}

def predownload(specs):
    code_ = "import segmentation_models_pytorch as smp, sys\nfor e in sys.argv[1:]: smp.Unet(e.split(':')[1], encoder_weights='imagenet')\nprint('weights ready')"
    print(subprocess.run([sys.executable, "-c", code_] + sorted(set(specs)), capture_output=True, text=True).stdout.strip())

def run_phase(jobs, name, deadline_ts, retries=2):
    path = f"{CFG['OUT']}/{name}_jobs.json"; json.dump(dict(cfg=WCFG, jobs=jobs), open(path, "w"))
    for attempt in range(retries + 1):
        todo = [j for j in jobs if not os.path.exists(f"{RES}/{j['id']}.npz")]
        if not todo or time.time() > deadline_ts: break
        for j in todo:
            lk = f"{RES}/{j['id']}.lock"
            if os.path.exists(lk): os.remove(lk)             # stale lock from a dead worker
        print(f"--- phase {name}: attempt {attempt + 1}, {len(todo)} jobs left, {NG} worker(s)")
        procs, logs = [], []
        for g in range(NG):
            lf = open(f"{CFG['OUT']}/{name}_gpu{g}_try{attempt}.log", "w"); logs.append(lf.name)
            procs.append(subprocess.Popen([sys.executable, "-u", WORKER, str(g), path, RES, CFG["CACHE"], str(deadline_ts)],
                                          env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g), "PYTHONUNBUFFERED": "1"}, stdout=lf, stderr=subprocess.STDOUT))
        pos = [0] * NG
        while True:
            alive = any(p.poll() is None for p in procs)
            for g, lg in enumerate(logs):
                with open(lg) as f: f.seek(pos[g]); new = f.read(); pos[g] = f.tell()
                for line in new.splitlines():
                    if line.strip(): print(f"[gpu{g}] {line}")
            if not alive: break
            time.sleep(20)
    done = [j for j in jobs if os.path.exists(f"{RES}/{j['id']}.npz")]
    print(f"phase {name}: {len(done)}/{len(jobs)} jobs finished | elapsed {hours():.2f} h"); return done

def chans_of(v):
    groups = CFG["VARIANTS"][v]; ch = list(GROUPS["base"])
    for gname in groups:
        if gname != "wx": ch += GROUPS[gname]
    return ch, ("wx" in groups)
def loss_of(v): return CFG["LOSS"].get(v, "bce_dice")
def jid(*a): return "__".join(str(x) for x in a).replace("+", "p").replace("|", "_").replace(":", "_")


job results available at start: 171
GPUs visible: 2 (Kaggle: choose 'GPU T4 x2' to use both)


In [9]:

# ---------- metrics helpers (parent process) ----------
def counts(prob, y, thr=0.5):
    m = y >= 0; pr = (prob >= thr) & m; t = (y == 1) & m
    return np.stack([(pr & t).sum((1, 2)), (pr & ~t).sum((1, 2)), (~pr & t).sum((1, 2))], 1)
def counts_bin(pr, y):
    m = y >= 0; pr = pr & m; t = (y == 1) & m
    return np.stack([(pr & t).sum((1, 2)), (pr & ~t).sum((1, 2)), (~pr & t).sum((1, 2))], 1)
def iou_c(c): return float(c[:, 0].sum() / max(c.sum(), 1)) if c.ndim == 2 else float(c[0] / max(c.sum(), 1))
def prec_c(c): return float(c[:, 0].sum() / max(c[:, 0].sum() + c[:, 1].sum(), 1))
def rec_c(c): return float(c[:, 0].sum() / max(c[:, 0].sum() + c[:, 2].sum(), 1))


In [10]:
# ---------- stats helpers ----------
def boot_pooled(cA, cB=None, n=2000, seed=0):
    rng = np.random.default_rng(seed); k = len(cA); a = []; d = []
    for _ in range(n):
        i = rng.integers(0, k, k); ia = iou_c(cA[i]); a.append(ia)
        if cB is not None: d.append(ia - iou_c(cB[i]))
    ci = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return (ci(a), (float(np.mean(d)), ci(d))) if cB is not None else ci(a)

def macro_iou(c, ev, ev_list):
    return float(np.mean([iou_c(c[ev == e]) for e in ev_list]))

def boot_macro(cA, cB, ev, ev_list, n=2000, seed=0):
    rng = np.random.default_rng(seed); groups = {e: np.where(ev == e)[0] for e in ev_list}; da = []; dd = []
    for _ in range(n):
        ma, mb = [], []
        for e in ev_list:
            g = rng.choice(groups[e], len(groups[e])); ma.append(iou_c(cA[g])); mb.append(iou_c(cB[g]))
        da.append(np.mean(ma)); dd.append(np.mean(ma) - np.mean(mb))
    ci = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return float(np.mean(dd)), ci(dd)

In [11]:

# ---------- stage 1: leave-one-event-out, 3 seeds, on 2 GPUs ----------
def boot_macro_multi(CA, CB, ev, ev_list, n=2000, seed=0):
    # CA/CB: lists (one per seed) of per-chip (tp, fp, fn); chips and events resampled jointly across seeds and variants
    rng = np.random.default_rng(seed); groups = {e: np.where(ev == e)[0] for e in ev_list}; dd = []
    for _ in range(n):
        gs = {e: rng.choice(groups[e], len(groups[e])) for e in ev_list}
        d = [np.mean([iou_c(a[gs[e]]) for e in ev_list]) - np.mean([iou_c(b[gs[e]]) for e in ev_list]) for a, b in zip(CA, CB)]
        dd.append(np.mean(d))
    return float(np.mean(dd)), (float(np.percentile(dd, 2.5)), float(np.percentile(dd, 97.5)))

events = EVENTS[:2] if CFG["SMOKE"] else EVENTS
loeo_jobs = []
for s in CFG["LOEO_SEEDS"]:
    for v in CFG["VARIANTS"]:
        ch, wxf = chans_of(v)
        for e in events:
            loeo_jobs.append(dict(id=jid("loeo", v, e, f"s{s}"), kind="loeo", variant=v, held=str(e), seed=s, enc=CFG["LOEO_ENC"],
                                  chans=ch, wx=wxf, loss=loss_of(v), steps=CFG["LOEO_STEPS"]))
print(len(loeo_jobs), "LOEO jobs (", len(CFG["LOEO_SEEDS"]), "seeds x", len(CFG["VARIANTS"]), "variants x", len(events), "events )")
predownload([CFG["LOEO_ENC"]] + CFG["FINAL_ENCODERS"])
done_l = run_phase(loeo_jobs, "loeo", T0 + 0.55 * CFG["DEADLINE_H"] * 3600)

PS, DONE = {}, {}
for j in done_l:
    d = np.load(f"{RES}/{j['id']}.npz"); key = (j["variant"], j["seed"])
    if key not in PS: PS[key] = np.zeros((N, S, S), np.float16); DONE[key] = np.zeros(N, bool)
    PS[key][d["idx"]] = d["p"]; DONE[key][d["idx"]] = True
V = list(CFG["VARIANTS"])
def seed_complete(s): return all((v, s) in DONE and all(DONE[(v, s)][EVENT == e].all() for e in events) for v in V)
seeds_use = [s for s in CFG["LOEO_SEEDS"] if seed_complete(s)]
assert seeds_use, "no seed finished every variant and event; rerun (finished jobs are kept)"
print("LOEO seeds complete:", seeds_use)
ev_done = list(events); hi = np.where(np.isin(EVENT, ev_done))[0]; evh = EVENT[hi]; Yh = Y[hi]; VARS = V
PROB = {v: np.mean([PS[(v, s)].astype(np.float32) for s in seeds_use], 0).astype(np.float16) for v in V}    # seed-ensembled held-out probabilities
CNT_S = {(v, s): counts(PS[(v, s)][hi].astype(np.float32), Yh, 0.5) for v in V for s in seeds_use}
CNT = {v: counts(PROB[v][hi].astype(np.float32), Yh, 0.5) for v in V}
rows = []
for v in V:
    ious = [macro_iou(CNT_S[(v, s)], evh, ev_done) for s in seeds_use]
    pr = {e: prec_c(CNT[v][evh == e]) for e in ev_done}; per = {e: iou_c(CNT[v][evh == e]) for e in ev_done}
    rows.append(dict(variant=v, macro_IoU_seed_mean=np.mean(ious), macro_IoU_seed_sd=(np.std(ious, ddof=1) if len(ious) > 1 else np.nan), per_seed=[round(x, 4) for x in ious],
                     seed_ensemble_macro_IoU=macro_iou(CNT[v], evh, ev_done), seed_ensemble_macro_precision=np.mean(list(pr.values())),
                     Somalia_prec=pr.get("Somalia", np.nan), Pakistan_prec=pr.get("Pakistan", np.nan), worst_event=min(per, key=per.get), worst_IoU=min(per.values()),
                     **{f"IoU_{e}": per[e] for e in ev_done}))
LOEO_DF = pd.DataFrame(rows).set_index("variant"); pd.set_option("display.width", 260, "display.max_columns", 50, "display.max_colwidth", 40)
print("\nLOEO (threshold 0.5): per-seed macro IoU, seed-ensemble result and per-event IoU:\n", LOEO_DF.round(3).T.to_string())
for v in V:
    if v == "S1": continue
    d, ci = boot_macro_multi([CNT_S[(v, s)] for s in seeds_use], [CNT_S[("S1", s)] for s in seeds_use], evh, ev_done)
    pdiff = [round(macro_iou(CNT_S[(v, s)], evh, ev_done) - macro_iou(CNT_S[("S1", s)], evh, ev_done), 4) for s in seeds_use]
    d2, ci2 = boot_macro(CNT[v], CNT["S1"], evh, ev_done)
    print(f"paired macro-IoU {v:>10} minus S1: seed-averaged {d:+.4f} 95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}] | per-seed diffs {pdiff} | seed-ensemble {d2:+.4f} [{ci2[0]:+.4f}, {ci2[1]:+.4f}]")
LOEO_DF.to_csv(f"{CFG['OUT']}/loeo_multiseed_by_event.csv")
BEST_INPUT = max([v for v in V if v != "S1"], key=lambda v: LOEO_DF.loc[v, "macro_IoU_seed_mean"]); BEST_LOSS = None
print("\nbest variant by seed-mean LOEO macro IoU:", BEST_INPUT)


132 LOEO jobs ( 3 seeds x 4 variants x 11 events )
weights ready
phase loeo: 132/132 jobs finished | elapsed 0.24 h
LOEO seeds complete: [0, 1, 2]

LOEO (threshold 0.5): per-seed macro IoU, seed-ensemble result and per-event IoU:
 variant                                              S1                    S1+ref                     S1+dw                 S1+ref+dw
macro_IoU_seed_mean                               0.564                     0.602                     0.585                     0.606
macro_IoU_seed_sd                                 0.001                     0.007                     0.008                     0.004
per_seed                       [0.5655, 0.5627, 0.5635]  [0.6075, 0.6048, 0.5941]  [0.5797, 0.5937, 0.5818]  [0.6026, 0.6043, 0.6102]
seed_ensemble_macro_IoU                           0.574                     0.617                       0.6                      0.62
seed_ensemble_macro_precision                     0.706                     0.768                  

In [12]:

# ---------- LOEO analysis: threshold sweeps, recall-matched precision, exclusion post-filters ----------
THR = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
def macro3(c, ev):
    return (float(np.mean([iou_c(c[ev == e]) for e in ev_done])), float(np.mean([prec_c(c[ev == e]) for e in ev_done])), float(np.mean([rec_c(c[ev == e]) for e in ev_done])))
PH = {v: PROB[v][hi].astype(np.float32) for v in VARS}
sweep = [dict(variant=v, thr=t, **dict(zip(("macro_IoU", "macro_precision", "macro_recall"), macro3(counts(PH[v], Yh, t), evh)))) for v in VARS for t in THR]
SW = pd.DataFrame(sweep)
for col in ("macro_IoU", "macro_precision", "macro_recall"):
    print(f"\n{col} by threshold:\n", SW.pivot(index="thr", columns="variant", values=col).round(3).to_string())

ref_row = SW[(SW.variant == "S1") & (SW.thr == 0.5)].iloc[0]; fine = np.arange(0.10, 0.951, 0.025); mrows = []
for v in VARS:
    best = None
    for t in fine:
        a, b, c_ = macro3(counts(PH[v], Yh, t), evh); d = abs(c_ - ref_row.macro_recall)
        if best is None or d < best[0]: best = (d, t, a, b, c_)
    mrows.append(dict(variant=v, thr=round(float(best[1]), 3), macro_recall=best[4], macro_precision=best[3], macro_IoU=best[2]))
print(f"\nRECALL-MATCHED (target macro recall {ref_row.macro_recall:.3f} = S1 at 0.5; reference precision {ref_row.macro_precision:.3f}, IoU {ref_row.macro_IoU:.4f}):")
print(pd.DataFrame(mrows).round(4).to_string(index=False))

# exclusion post-filters: terrain (HAND, slope) and Dynamic World land cover, fixed grid, trend only
HANDH = np.stack([(TERR[IDS[i]][3] if TERR[IDS[i]] is not None else np.zeros((S, S), np.float32)) for i in hi]); SLOPEH = np.stack([(TERR[IDS[i]][2] if TERR[IDS[i]] is not None else np.zeros((S, S), np.float32)) for i in hi])
DWH = np.stack([(np.load(dw_path(IDS[i])).astype(np.float32) if os.path.exists(dw_path(IDS[i])) else np.zeros((9, S, S), np.float32)) for i in hi])
bare, built = DWH[:, 7], DWH[:, 6]
EXCL = {"none": np.zeros_like(HANDH, bool), "HAND>15": HANDH > 15, "HAND>15|slope>5": (HANDH > 15) | (SLOPEH > 5),
        "DW bare>0.5": bare > 0.5, "DW built>0.5": built > 0.5, "DW bare|built>0.5": (bare > 0.5) | (built > 0.5),
        "HAND>15|slope>5|bare>0.5": (HANDH > 15) | (SLOPEH > 5) | (bare > 0.5)}
erows = []
for v in VARS:
    for nme, mk in EXCL.items():
        a, b, c_ = macro3(counts_bin((PH[v] >= 0.5) & ~mk, Yh), evh); erows.append(dict(variant=v, exclusion=nme, macro_IoU=a, macro_precision=b, macro_recall=c_, removed_share=float(mk.mean())))
EX = pd.DataFrame(erows)
for col in ("macro_IoU", "macro_precision", "macro_recall"):
    print(f"\nexclusion filters, {col} (threshold 0.5):\n", EX.pivot(index="exclusion", columns="variant", values=col).round(3).to_string())
SW.to_csv(f"{CFG['OUT']}/loeo_threshold_sweep.csv", index=False); EX.to_csv(f"{CFG['OUT']}/loeo_exclusion.csv", index=False); pd.DataFrame(mrows).to_csv(f"{CFG['OUT']}/loeo_recall_matched.csv", index=False)
del PH, DWH, HANDH, SLOPEH



macro_IoU by threshold:
 variant     S1  S1+dw  S1+ref  S1+ref+dw
thr                                     
0.3      0.569  0.597   0.605      0.613
0.4      0.574  0.601   0.615      0.620
0.5      0.574  0.600   0.617      0.620
0.6      0.569  0.590   0.610      0.612
0.7      0.552  0.572   0.586      0.595
0.8      0.516  0.542   0.543      0.569
0.9      0.446  0.476   0.470      0.497

macro_precision by threshold:
 variant     S1  S1+dw  S1+ref  S1+ref+dw
thr                                     
0.3      0.652  0.721   0.703      0.740
0.4      0.680  0.759   0.738      0.776
0.5      0.706  0.790   0.768      0.805
0.6      0.732  0.818   0.799      0.832
0.7      0.759  0.843   0.829      0.857
0.8      0.790  0.868   0.857      0.885
0.9      0.836  0.898   0.898      0.918

macro_recall by threshold:
 variant     S1  S1+dw  S1+ref  S1+ref+dw
thr                                     
0.3      0.809  0.771   0.802      0.783
0.4      0.776  0.740   0.775      0.755
0.5      0.

In [13]:

# ---------- stage 2: official split, 3 encoders x 5 seeds, train split only (standard protocol) ----------
FINAL_VARIANTS = list(CFG["VARIANTS"])
final_jobs = []
for s in CFG["FINAL_SEEDS"]:
    for v in FINAL_VARIANTS:
        ch, wxf = chans_of(v)
        for enc in CFG["FINAL_ENCODERS"]:
            final_jobs.append(dict(id=jid("final", v, enc, f"s{s}"), kind="final", variant=v, seed=s, enc=enc, chans=ch, wx=wxf, loss=loss_of(v), steps=CFG["FINAL_STEPS"]))
print("official-split jobs:", len(final_jobs))
done_f = run_phase(final_jobs, "final", T0 + 0.85 * CFG["DEADLINE_H"] * 3600)
FINAL = []
for j in done_f:
    d = np.load(f"{RES}/{j['id']}.npz"); r = dict(variant=j["variant"], enc=j["enc"], seed=j["seed"], best_valid=float(d["best_valid"]), probs={k: d[k] for k in ("valid", "test", "bol")})
    for k in ("test", "bol"): r[k + "_iou"] = iou_c(counts(r["probs"][k].astype(np.float32), Y[IDX[k]]))
    FINAL.append(r)
assert FINAL, "no final model finished"


official-split jobs: 60
--- phase final: attempt 1, 33 jobs left, 2 worker(s)
[gpu0] worker on GPU 0 | jobs 60
[gpu0] /kaggle/working/worker.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
[gpu0]   scaler = torch.cuda.amp.GradScaler()
[gpu1] worker on GPU 1 | jobs 60
[gpu1] /kaggle/working/worker.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
[gpu1]   scaler = torch.cuda.amp.GradScaler()
[gpu0] [final__S1pref__unet_resnet34__s2] step 250/3000 loss 1.2433 (53s) valid IoU(EMA) 0.6296
[gpu1] [final__S1pref__unet_resnext50_32x4d__s2] step 250/3000 loss 1.1489 (79s) valid IoU(EMA) 0.6313
[gpu0] [final__S1pref__unet_resnet34__s2] step 500/3000 loss 0.8818 (106s) valid IoU(EMA) 0.6523
[gpu0] [final__S1pref__unet_resnet34__s2] step 750/3000 loss 0.7839 (160s) valid IoU(EMA) 0.6775
[gpu1] [final__S1pref__unet_resnext50_32x

In [14]:

# ---------- official-split results ----------
def ens(runs, split): return np.mean([r["probs"][split].astype(np.float32) for r in runs], 0)
def best_thr(runs):
    pv = ens(runs, "valid"); grid = np.arange(0.30, 0.71, 0.02)
    return float(grid[int(np.argmax([iou_c(counts(pv, Y[IDX["valid"]], t)) for t in grid]))])
def report(name, runs, thr=None):
    thr = best_thr(runs) if thr is None else thr; row = dict(model=name, n=len(runs), thr=round(thr, 2))
    for sp in ["valid", "test", "bol"]:
        c = counts(ens(runs, sp), Y[IDX[sp]], thr); row[sp] = iou_c(c)
        if sp != "valid": row[sp + "_ci95"] = tuple(round(x, 3) for x in boot_pooled(c)); row[sp + "_prec"] = prec_c(c); row[sp + "_rec"] = rec_c(c)
    return row
rows = []
for v in FINAL_VARIANTS:
    for enc in CFG["FINAL_ENCODERS"]:
        sub = [r for r in FINAL if r["variant"] == v and r["enc"] == enc]
        if sub: rows.append(report(f"{v} | {enc} x{len(sub)}", sub))
    rows.append(report(f"{v} | ALL ensemble", [r for r in FINAL if r["variant"] == v]))
RES_DF = pd.DataFrame(rows); print(RES_DF.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
singles = pd.DataFrame([{k: r[k] for k in ["variant", "enc", "seed", "best_valid", "test_iou", "bol_iou"]} for r in FINAL])
print("\nsingle models:\n", singles.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
base_runs = [r for r in FINAL if r["variant"] == "S1"]
for v in FINAL_VARIANTS:
    if v == "S1": continue
    a = [r for r in FINAL if r["variant"] == v]
    if not a or not base_runs: continue
    ta, tb = best_thr(a), best_thr(base_runs)
    ca, cb = counts(ens(a, "test"), Y[IDX["test"]], ta), counts(ens(base_runs, "test"), Y[IDX["test"]], tb)
    _, (dm, dci) = boot_pooled(ca, cb)
    print(f"\nTEST paired (same chips): {v} {iou_c(ca):.4f} vs S1 {iou_c(cb):.4f} | diff {dm:+.4f} 95% CI [{dci[0]:+.4f}, {dci[1]:+.4f}] | precision {prec_c(ca):.3f} vs {prec_c(cb):.3f} | recall {rec_c(ca):.3f} vs {rec_c(cb):.3f}")
print("\nReference: DeepSARFlood reports 0.72 test IoU (single-date S1 + JRC/DEM/slope/HAND inputs). Variants with pre-event reference, land cover or weather are multi-source/bi-temporal: label them as such. Bi-temporal prior work on Sen1Floods11: Yadav et al., IGARSS 2022 (arXiv 2204.09387).")
RES_DF.to_csv(f"{CFG['OUT']}/official_split_ensembles.csv", index=False); singles.to_csv(f"{CFG['OUT']}/single_models.csv", index=False)
json.dump(dict(cfg=CFG, best_input=BEST_INPUT, best_loss=BEST_LOSS, elapsed_h=hours(), gpus=NG), open(f"{CFG['OUT']}/run.json", "w"), indent=1, default=str)
print("saved to", CFG["OUT"], "| total %.2f h" % hours())


                              model  n    thr  valid   test      test_ci95  test_prec  test_rec    bol       bol_ci95  bol_prec  bol_rec
              S1 | unet:resnet34 x5  5 0.4400 0.6693 0.6883  (0.58, 0.771)     0.8213    0.8096 0.5897 (0.421, 0.692)    0.7896   0.6996
       S1 | unet:resnext50_32x4d x5  5 0.4800 0.6714 0.6893  (0.58, 0.772)     0.8241    0.8082 0.5718 (0.409, 0.667)    0.7736   0.6867
                S1 | unet:mit_b2 x5  5 0.4600 0.6736 0.6771 (0.565, 0.763)     0.8198    0.7955 0.6281 (0.438, 0.742)    0.7795   0.7638
                  S1 | ALL ensemble 15 0.4600 0.6780 0.6897 (0.578, 0.774)     0.8262    0.8067 0.6060 (0.437, 0.704)    0.7882   0.7239
          S1+ref | unet:resnet34 x5  5 0.4400 0.6949 0.7074 (0.607, 0.784)     0.8330    0.8242 0.6858 (0.481, 0.788)    0.7257   0.9258
   S1+ref | unet:resnext50_32x4d x5  5 0.4600 0.7064 0.7116 (0.614, 0.789)     0.8462    0.8174 0.6878 (0.485, 0.788)    0.7425   0.9034
            S1+ref | unet:mit_b2 x5  5 0.

In [15]:

# ---------- extra: train+valid fit (NON-STANDARD protocol, reported separately) ----------
full_jobs = []
for s in CFG["FULL_SEEDS"]:
    for v in CFG["FULL_VARIANTS"]:
        ch, wxf = chans_of(v)
        for enc in CFG["FINAL_ENCODERS"]:
            full_jobs.append(dict(id=jid("full", v, enc, f"s{s}"), kind="final_full", variant=v, seed=s, enc=enc, chans=ch, wx=wxf, loss=loss_of(v), steps=CFG["FINAL_STEPS"]))
print("train+valid jobs:", len(full_jobs))
done_u = run_phase(full_jobs, "full", T0 + CFG["DEADLINE_H"] * 3600)
FULL = []
for j in done_u:
    d = np.load(f"{RES}/{j['id']}.npz"); FULL.append(dict(variant=j["variant"], enc=j["enc"], seed=j["seed"], probs={k: d[k] for k in ("valid", "test", "bol")}))
frows, cnt_full = [], {}
for v in CFG["FULL_VARIANTS"]:
    runs = [r for r in FULL if r["variant"] == v]; base = [r for r in FINAL if r["variant"] == v]
    if not runs or not base: continue
    thr = best_thr(base)                                   # threshold from the train-only ensemble on valid; nothing is tuned on test
    row = dict(variant=v, n_models=len(runs), thr=round(thr, 2))
    for sp in ("test", "bol"):
        c = counts(ens(runs, sp), Y[IDX[sp]], thr); row[sp] = iou_c(c); row[sp + "_ci95"] = tuple(round(x, 3) for x in boot_pooled(c))
        row[sp + "_prec"] = prec_c(c); row[sp + "_rec"] = rec_c(c)
        if sp == "test": cnt_full[v] = c
    frows.append(row)
if frows:
    FULL_DF = pd.DataFrame(frows); print("\nTRAIN+VALID ensembles (validation chips are in training; valid IoU not reported):\n", FULL_DF.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    if "S1" in cnt_full and "S1+ref+dw" in cnt_full:
        _, (dm, dci) = boot_pooled(cnt_full["S1+ref+dw"], cnt_full["S1"])
        print(f"TEST paired, both trained on train+valid: S1+ref+dw minus S1 = {dm:+.4f}  95% CI [{dci[0]:+.4f}, {dci[1]:+.4f}]")
    FULL_DF.to_csv(f"{CFG['OUT']}/train_plus_valid_ensembles.csv", index=False)
print("\nReport the train+valid numbers separately from the standard-protocol ones (more training data).")
print("total %.2f h" % hours())


train+valid jobs: 12
phase full: 12/12 jobs finished | elapsed 4.38 h

TRAIN+VALID ensembles (validation chips are in training; valid IoU not reported):
   variant  n_models    thr   test      test_ci95  test_prec  test_rec    bol       bol_ci95  bol_prec  bol_rec
       S1         6 0.4600 0.7059 (0.596, 0.788)     0.8470    0.8090 0.6020 (0.438, 0.686)    0.7396   0.7638
S1+ref+dw         6 0.4400 0.7160 (0.613, 0.793)     0.8607    0.8098 0.6950 (0.506, 0.801)    0.7675   0.8803
TEST paired, both trained on train+valid: S1+ref+dw minus S1 = +0.0099  95% CI [-0.0437, +0.0649]

Report the train+valid numbers separately from the standard-protocol ones (more training data).
total 4.38 h


## How to read it
1. **Multi-seed LOEO first:** the seed-averaged paired difference and its interval are the main evidence. Check that all per-seed differences have the same sign.
2. **Official split (15-model ensembles):** the paired test difference still has a wide interval (90 chips). Use it as supporting evidence only.
3. **Train+valid** results are a separate "more data" row: labelled non-standard, never compared directly with published train-split-only numbers.
4. Save the whole output folder; `jobs_results/*.npz` lets you resume or re-analyse without retraining.